In [ ]:
import sys
import os
import pygame

# --- PyQt5 Imports ---
from PyQt5.QtWidgets import QApplication, QWidget, QVBoxLayout, QLabel, QProgressBar
from PyQt5.QtCore import QTimer

class SteamDeckController(QWidget):
    def __init__(self):
        super().__init__()
        
        # --- 1. Setup Pygame for Controller (Headless Mode) ---
        # This prevents Pygame from opening a black window, since we only need input.
        os.environ["SDL_VIDEODRIVER"] = "dummy"
        pygame.init()
        pygame.joystick.init()

        self.controller = None
        self.connect_controller()

        # --- 2. Setup PyQt5 GUI ---
        self.initUI()

        # --- 3. Setup the Polling Timer ---
        # A standard GUI waits for clicks. A controller is always moving.
        # We use a QTimer to check the controller state 30 times a second.
        self.timer = QTimer()
        self.timer.timeout.connect(self.update_loop)
        self.timer.start(33) # 33ms is approx 30 frames per second

    def connect_controller(self):
        # Check if any controller is connected
        if pygame.joystick.get_count() > 0:
            self.controller = pygame.joystick.Joystick(0)
            self.controller.init()
            print(f"Connected to: {self.controller.get_name()}")
        else:
            print("No controller found! Ensure you are in Desktop Mode with Steam running.")

    def initUI(self):
        layout = QVBoxLayout()

        # Visualizers
        self.lbl_status = QLabel("Status: Scanning...", self)
        
        # Helper instruction
        self.lbl_instruction = QLabel("Move Left Stick to see values.", self)

        # Labels for Data
        self.lbl_left_stick = QLabel("Left Stick (Robot Drive): (0.00, 0.00)", self)
        self.lbl_buttons = QLabel("Buttons Pressed: None", self)
        
        # Progress bar for a trigger (e.g., speed boost or arm gripper)
        self.lbl_trigger = QLabel("Right Trigger (Gripper):", self)
        self.trigger_bar = QProgressBar(self)
        self.trigger_bar.setRange(0, 100)
        self.trigger_bar.setValue(0)

        # Add widgets to layout
        layout.addWidget(self.lbl_status)
        layout.addWidget(self.lbl_instruction)
        layout.addWidget(self.lbl_left_stick)
        layout.addWidget(self.lbl_buttons)
        layout.addWidget(self.lbl_trigger)
        layout.addWidget(self.trigger_bar)

        self.setLayout(layout)
        self.setWindowTitle('Steam Deck Robot Controller (PyQt5)')
        self.resize(400, 250)

    def update_loop(self):
        """
        This function runs every 33ms to read the hardware state.
        """
        # A. Process Pygame Events (Essential to keep state updated)
        pygame.event.pump() 

        if not self.controller:
            self.lbl_status.setText("Status: No Controller Found")
            return

        self.lbl_status.setText(f"Status: Connected to {self.controller.get_name()}")

        # --- B. READ AXIS (Left Stick) ---
        # Axis 0 = Left/Right, Axis 1 = Up/Down
        raw_x = self.controller.get_axis(0)
        raw_y = self.controller.get_axis(1)

        # DEADZONE LOGIC:
        # Controllers drift. If the stick is close to center (< 0.1), treat it as 0.
        DEADZONE = 0.1
        
        left_x = 0 if abs(raw_x) < DEADZONE else raw_x
        left_y = 0 if abs(raw_y) < DEADZONE else raw_y
        
        self.lbl_left_stick.setText(f"Left Stick (Robot Drive): ({left_x:.2f}, {left_y:.2f})")

        # --- C. READ TRIGGERS (Right Trigger) ---
        # Usually Axis 5 on Linux/Steam Deck. Returns -1.0 (out) to 1.0 (in).
        trigger_val = self.controller.get_axis(5) 
        
        # Map from [-1.0 ... 1.0] range to [0 ... 100] range for the progress bar
        # (value + 1) makes it 0 to 2. * 50 makes it 0 to 100.
        mapped_trigger = int((trigger_val + 1) * 50)
        self.trigger_bar.setValue(mapped_trigger)

        # --- D. READ BUTTONS ---
        pressed_buttons = []
        # Iterate through first 10 buttons
        for i in range(10): 
            if self.controller.get_button(i):
                # You can map these indices to names later (0 is usually 'A' or 'X')
                pressed_buttons.append(f"Btn {i}")
        
        if pressed_buttons:
            self.lbl_buttons.setText(f"Buttons: {', '.join(pressed_buttons)}")
        else:
            self.lbl_buttons.setText("Buttons: None")

if __name__ == '__main__':
    app = QApplication(sys.argv)
    ex = SteamDeckController()
    ex.show()
    # PyQt5 uses exec_() with an underscore
    sys.exit(app.exec_())